In [30]:
import argparse
import os,cv2
import random,numpy,pandas
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [31]:
import argparse
import os
import random,numpy
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
import torchvision.utils as vutils
from torchvision.models.feature_extraction import create_feature_extractor
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [32]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

Random Seed:  999


In [33]:
fsc_test={}
for i in os.listdir('/kaggle/input/cidaut-ai-fake-scene-classification-2024/Test'):
    fsc_test[i]=TF.to_tensor(cv2.resize(cv2.imread(f'/kaggle/input/cidaut-ai-fake-scene-classification-2024/Test/{i}'),(256, 256)))

print(fsc_test["21.jpg"].shape)
device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

torch.Size([3, 256, 256])


In [34]:
class EffnetModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        effnet = torchvision.models.efficientnet_v2_m(weights=torchvision.models.efficientnet.EfficientNet_V2_M_Weights.DEFAULT)
        self.model = create_feature_extractor(effnet, ['flatten'])
        self.nn_fracture = torch.nn.Sequential(
            torch.nn.Linear(1280, 2),
        )
    def forward(self, x):
        x = self.model(x)['flatten']
        x = self.nn_fracture(x)
        return torch.argmax(x, dim=1)

In [35]:
EFF_NET = EffnetModel().float().to(device)
#EFF_NET

#EFF_NET.load_state_dict(torch.load("/kaggle/input/d/mafiosoquasar/fake-scene-classification-model-2/0.00015554654237348586 87.0.pth",weights_only=False,map_location=torch.device('cpu')))


In [ ]:
fsc_submission=pandas.read_csv("/kaggle/input/cidaut-ai-fake-scene-classification-2024/sample_submission.csv", index_col ="image")#.to_dict('list')
sub={}

for j in ['1.8978218463416852e-07 89.0']:
    z_add,z_total=0,0
    EFF_NET.load_state_dict(torch.load(f"/kaggle/input/d/mafiosoquasar/fake-scene-classification-model-2/{j}.pth",weights_only=False,map_location=torch.device('cpu')))
    for i in fsc_submission.index:
        img=fsc_test[i].reshape((1, 3, 256, 256)).float().to(device)
        #print(FSC_net(img))

        output = EFF_NET(img).cpu().detach().numpy()[0]
        if output==0:
            z_add+=1
        z_total+=1
        #fsc_submission.loc[i]['label']=output
        if i not in sub.keys():
            if output==0:
                sub[i]={0:1,
                       1:0}
            else:
                sub[i]={0:0,
                       1:1}
        else:
            if output==0:
                sub[i][0]+=1
            else:
                sub[i][1]+=1
    print(z_total,z_add)
    
for i in fsc_submission.index:
    sub[i]=list(sub[i].values())

In [ ]:
fsc_submission=pandas.read_csv("/kaggle/input/cidaut-ai-fake-scene-classification-2024/sample_submission.csv", index_col ="image")#.to_dict('list')
z_add,z_total=0,0

for i in fsc_submission.index:
    if sub[i][0]!=0:
        fsc_submission.loc[i]['label']=0
        z_add+=1
    else:
        fsc_submission.loc[i]['label']=1
    z_total+=1
print(z_total,z_add)

pandas.DataFrame(fsc_submission)

In [ ]:
fsc_submission['image']=fsc_submission.index
pandas.DataFrame(fsc_submission).to_csv(f"submission.csv", index=False)